# SDialog dependencies

In [ ]:
# Setup the environment depending on weather we are running in Google Colab or Jupyter Notebook
import os
from IPython import get_ipython

if "google.colab" in str(get_ipython()):
    print("Running on CoLab")

    # Installing sdialog
    !git clone https://github.com/idiap/sdialog.git
    %cd sdialog
    %pip install -e .
    %cd ..
else:
    print("Running in Jupyter Notebook")
    # Little hack to avoid the "OSError: Background processes not supported." error in Jupyter notebooks"
    get_ipython().system = os.system

## Local installation

Create a `.venv` using the root `requirement.txt` file and Python `3.11.14`

# Tutorial 17: Generating data with new models

In [ ]:
os.environ["AWS_BEARER_TOKEN_BEDROCK"] = "XXX"

In [ ]:
import os
import sdialog
from sdialog import Context 
from sdialog.agents import Agent
from sdialog.personas import Persona

In [ ]:
model_info = {
    "path": "us.meta.llama4-scout-17b-instruct-v1:0",
    "file_name": "Llama4Scout"
}
# model_info = {
#     "path": "us.anthropic.claude-sonnet-4-5-20250929-v1:0",
#     "file_name": "Sonnet45"
# }

In [ ]:
sdialog.config.llm(
    model_info["path"], 
    temperature=0.7, 
    max_tokens=512,
    region_name="us-east-1"
)

In [ ]:
patient = Persona(
    name="Marie Dubois",
    age=45,
    gender="female",
    role="patient",
    background="Marie is a 45-year-old patient who is consulting for recurring headaches and fatigue. She works in an office and has two teenage children.",
    personality="Marie is polite, a bit anxious, and tends to ask many questions. She worries easily about her health.",
    circumstances="She made an appointment because her symptoms have persisted for two weeks and are affecting her work.",
    rules="She must be respectful towards the doctor and express her concerns clearly."
)

doctor = Persona(
    name="Dr. Pierre Martin",
    age=52,
    gender="male", 
    role="general practitioner",
    background="Dr. Martin is an experienced general practitioner with 25 years of experience. He is known for his empathetic approach and patience with his patients.",
    personality="He is very professional, empathetic, and takes time to listen to his patients. He explains things clearly and reassuringly.",
    circumstances="He practices in his private office and has a holistic approach to medicine.",
    rules="He must be polite, professional, ask relevant questions, and reassure the patient while remaining medically precise."
)

In [ ]:
context = Context(
    location="Dr. Martin's medical office",
    datetime="2024-01-15 14:30",
    environment="A modern and welcoming medical office with a calm and professional atmosphere",
    topics=["medical consultation", "headaches", "fatigue", "diagnosis"],
    goals=["establish a diagnosis", "reassure the patient", "propose treatment"],
    constraints=["respect medical confidentiality", "be professional", "limit consultation to 20 minutes"]
)

In [ ]:
agent_patient = Agent(
    persona=patient,
    name="Marie",
    context=context,
    first_utterance="Hello doctor, thank you for seeing me. I came to see you because I've had persistent headaches for two weeks, and I feel very tired.",
    dialogue_details="Medical consultation for persistent symptoms",
    response_details="Natural and respectful responses, expressing patient concerns"
)

agent_doctor = Agent(
    persona=doctor,
    name="Dr. Martin", 
    context=context,
    dialogue_details="Professional medical consultation",
    response_details="Professional, empathetic and reassuring medical responses"
)

In [ ]:
dialogue = agent_patient.dialog_with(
    agent_doctor,
    context=context,
    max_turns=12,  # 6 exchanges (12 turns total)
    seed=42  # For reproducibility
)

dialogue.print()

In [ ]:
print(f"\n=== Dialog Statistic ===")
print(f"Number of turns: {len(dialogue)}")
print(f"Number of words: {dialogue.length('words')}")
print(f"Estimated duration: {dialogue.length('minutes')} minutes")
print(f"Model used: {dialogue.model}")
print(f"Seed: {dialogue.seed}")

In [ ]:
dialogue.to_file(f"demo_dialog_doctor_patient_with_LLM={model_info['file_name']}.json")